In [6]:
import ssl
import nltk
from sklearn.model_selection import train_test_split

# Desactivar verificación SSL temporalmente
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Descarga de recursos requeridos por NLTK
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/josetanchezz/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
file_path = '/Users/josetanchezz/Desktop/NLP/Lab5/don-quijote.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

sentences = nltk.sent_tokenize(text, language='spanish')

tokenized_sentences = []
for sent in sentences:
    tokens = nltk.word_tokenize(sent, language='spanish')
    tokenized_sentences.append(['<s>'] + tokens + ['</s>'])

print(f"Total de oraciones procesadas: {len(tokenized_sentences)}")


Total de oraciones procesadas: 9673


In [8]:
# División aleatoria: Entrenamiento (80%), Validación (10%), Prueba (10%)
train_sentences, temp_sentences = train_test_split(tokenized_sentences, test_size=0.2, random_state=42)
val_sentences, test_sentences = train_test_split(temp_sentences, test_size=0.5, random_state=42)

print(f"Oraciones en Entrenamiento: {len(train_sentences)}")
print(f"Oraciones en Validación: {len(val_sentences)}")
print(f"Oraciones en Prueba: {len(test_sentences)}")


Oraciones en Entrenamiento: 7738
Oraciones en Validación: 967
Oraciones en Prueba: 968


In [9]:
# Creación del vocabulario a partir del conjunto de entrenamiento
train_vocab = set(word for sent in train_sentences for word in sent)
train_vocab_size = len(train_vocab)

# Extracción de palabras del conjunto de prueba
test_words = [word for sent in test_sentences for word in sent]
total_test_words = len(test_words)

# Cálculo de la proporción de palabras OOV
oov_words = [word for word in test_words if word not in train_vocab]
oov_proportion = len(oov_words) / total_test_words if total_test_words > 0 else 0

print(f"Tamaño del vocabulario de entrenamiento: {train_vocab_size}")
print(f"Proporción de palabras OOV en prueba: {oov_proportion:.2%}")


Tamaño del vocabulario de entrenamiento: 23305
Proporción de palabras OOV en prueba: 3.50%


In [10]:
from collections import defaultdict, Counter

# Estructuras de almacenamiento mediante diccionarios y contadores
unigram_counts = Counter()
bigram_counts = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

total_unigrams = 0

# Conteo de n-gramas sobre el conjunto de entrenamiento
for sent in train_sentences:
    for w in sent:
        unigram_counts[w] += 1
        total_unigrams += 1
    
    for w1, w2 in zip(sent[:-1], sent[1:]):
        bigram_counts[w1][w2] += 1
        
    for w1, w2, w3 in zip(sent[:-2], sent[1:-1], sent[2:]):
        trigram_counts[(w1, w2)][w3] += 1

print(f"Total de unigramas contados: {total_unigrams}")
print(f"Unigramas únicos: {len(unigram_counts)}")
print(f"Contextos de bigrama únicos: {len(bigram_counts)}")
print(f"Contextos de trigrama únicos: {len(trigram_counts)}")

Total de unigramas contados: 369938
Unigramas únicos: 23305
Contextos de bigrama únicos: 23304
Contextos de trigrama únicos: 130558


In [11]:
def get_unigram_prob(word):
    return unigram_counts[word] / total_unigrams if total_unigrams > 0 else 0.0

def get_bigram_prob(w_prev, w_curr):
    context_count = sum(bigram_counts[w_prev].values())
    if context_count == 0:
        return 0.0
    return bigram_counts[w_prev][w_curr] / context_count

def get_trigram_prob(w_prev2, w_prev1, w_curr):
    context_count = sum(trigram_counts[(w_prev2, w_prev1)].values())
    if context_count == 0:
        return 0.0
    return trigram_counts[(w_prev2, w_prev1)][w_curr] / context_count


In [12]:
# Selección de una oración de ejemplo del conjunto de validación
sample_sentence = val_sentences[0]

# 1. Probabilidad Unigrama
p_unigram = 1.0
for w in sample_sentence:
    p_unigram *= get_unigram_prob(w)

# 2. Probabilidad Bigrama
p_bigram = 1.0
for w1, w2 in zip(sample_sentence[:-1], sample_sentence[1:]):
    p_bigram *= get_bigram_prob(w1, w2)

# 3. Probabilidad Trigrama
p_trigram = 1.0
if len(sample_sentence) > 1:
    p_trigram *= get_bigram_prob(sample_sentence[0], sample_sentence[1])
    for w1, w2, w3 in zip(sample_sentence[:-2], sample_sentence[1:-1], sample_sentence[2:]):
        p_trigram *= get_trigram_prob(w1, w2, w3)

print("Oración evaluada:", " ".join(sample_sentence))
print(f"Probabilidad Unigrama: {p_unigram}")
print(f"Probabilidad Bigrama: {p_bigram}")
print(f"Probabilidad Trigrama: {p_trigram}")


Oración evaluada: <s> Para mi santiguada , que tengo por cierto que si Reinaldos de Montalbán hubiera oído estas razones al hombrecito , tapaboca le hubiera dado que no hablara más en tres años . </s>
Probabilidad Unigrama: 0.0
Probabilidad Bigrama: 0.0
Probabilidad Trigrama: 0.0


In [13]:
# Tamaño del vocabulario de entrenamiento |V|
vocab_size = len(train_vocab)

def get_bigram_prob_add_k(w_prev, w_curr, k=1.0):
    context_count = sum(bigram_counts[w_prev].values())
    bigram_count = bigram_counts[w_prev][w_curr]
    
    numerator = bigram_count + k
    denominator = context_count + (k * vocab_size)
    
    return numerator / denominator

In [14]:
import math

# Evaluación de la oración de prueba con distintos valores de k
p_no_smooth = p_bigram
p_laplace = 1.0     # k = 1.0
p_add_k_01 = 1.0    # k = 0.1
p_add_k_001 = 1.0   # k = 0.01

log_p_laplace = 0.0
log_p_add_k_01 = 0.0
log_p_add_k_001 = 0.0

for w1, w2 in zip(sample_sentence[:-1], sample_sentence[1:]):
    prob_lap = get_bigram_prob_add_k(w1, w2, k=1.0)
    prob_k01 = get_bigram_prob_add_k(w1, w2, k=0.1)
    prob_k001 = get_bigram_prob_add_k(w1, w2, k=0.01)
    
    p_laplace *= prob_lap
    p_add_k_01 *= prob_k01
    p_add_k_001 *= prob_k001
    
    log_p_laplace += math.log(prob_lap)
    log_p_add_k_01 += math.log(prob_k01)
    log_p_add_k_001 += math.log(prob_k001)

print(f"Probabilidad SIN suavizado: {p_no_smooth}")
print(f"Probabilidad Laplace (k=1.0): {p_laplace} (log-prob: {log_p_laplace:.4f})")
print(f"Probabilidad Add-k (k=0.1): {p_add_k_01} (log-prob: {log_p_add_k_01:.4f})")
print(f"Probabilidad Add-k (k=0.01): {p_add_k_001} (log-prob: {log_p_add_k_001:.4f})")

Probabilidad SIN suavizado: 0.0
Probabilidad Laplace (k=1.0): 3.865451576641467e-119 (log-prob: -272.6555)
Probabilidad Add-k (k=0.1): 2.2595915868359472e-107 (log-prob: -245.5614)
Probabilidad Add-k (k=0.01): 1.064257628842052e-101 (log-prob: -232.4988)


In [15]:
import math

# Pre-cálculo de la suma de frecuencias de contexto para optimizar la velocidad
bigram_context_counts = {w_prev: sum(counts.values()) for w_prev, counts in bigram_counts.items()}
trigram_context_counts = {context: sum(counts.values()) for context, counts in trigram_counts.items()}

def get_unigram_prob_smoothed(word, k=1.0):
    return (unigram_counts[word] + k) / (total_unigrams + k * vocab_size)

def get_bigram_prob_smoothed(w_prev, w_curr, k=1.0):
    context_count = bigram_context_counts.get(w_prev, 0)
    return (bigram_counts[w_prev][w_curr] + k) / (context_count + k * vocab_size)

def get_trigram_prob_smoothed(w_prev2, w_prev1, w_curr, k=1.0):
    context_count = trigram_context_counts.get((w_prev2, w_prev1), 0)
    return (trigram_counts[(w_prev2, w_prev1)][w_curr] + k) / (context_count + k * vocab_size)

def calculate_perplexity(dataset, model_type='bigram', k=1.0):
    total_log_prob = 0.0
    total_words = 0
    
    for sent in dataset:
        if model_type == 'unigram':
            for w in sent:
                prob = get_unigram_prob_smoothed(w, k=k)
                total_log_prob += math.log(prob)
                total_words += 1
        elif model_type == 'bigram':
            for w1, w2 in zip(sent[:-1], sent[1:]):
                prob = get_bigram_prob_smoothed(w1, w2, k=k)
                total_log_prob += math.log(prob)
                total_words += 1
        elif model_type == 'trigram':
            if len(sent) > 1:
                prob = get_bigram_prob_smoothed(sent[0], sent[1], k=k)
                total_log_prob += math.log(prob)
                total_words += 1
            for w1, w2, w3 in zip(sent[:-2], sent[1:-1], sent[2:]):
                prob = get_trigram_prob_smoothed(w1, w2, w3, k=k)
                total_log_prob += math.log(prob)
                total_words += 1
                
    if total_words == 0:
        return float('inf')
    return math.exp(-total_log_prob / total_words)


In [17]:
import pandas as pd

k_eval = 0.1

pp_unigram_val = calculate_perplexity(val_sentences, model_type='unigram', k=k_eval)
pp_bigram_val = calculate_perplexity(val_sentences, model_type='bigram', k=k_eval)
pp_trigram_val = calculate_perplexity(val_sentences, model_type='trigram', k=k_eval)

df_pp = pd.DataFrame({
    'Modelo': ['Unigrama', 'Bigrama', 'Trigrama'],
    'Perplejidad Validación (k=0.1)': [pp_unigram_val, pp_bigram_val, pp_trigram_val]
})

print(df_pp.to_string(index=False))


  Modelo  Perplejidad Validación (k=0.1)
Unigrama                      638.047105
 Bigrama                      810.283126
Trigrama                     5186.617878


In [18]:
# 1. Elección del mejor modelo según validación y prueba final
best_model_name = 'bigram' if pp_bigram_val <= pp_trigram_val else 'trigram'
pp_test = calculate_perplexity(test_sentences, model_type=best_model_name, k=k_eval)

print(f"Mejor modelo según validación: {best_model_name.capitalize()}")
print(f"Perplejidad final en el conjunto de Prueba: {pp_test:.2f}\n")

# 2. Análisis del impacto de k en el modelo de Bigrama
k_values = [1.0, 0.5, 0.1, 0.01, 0.001]
k_experiments = []

for k in k_values:
    pp_k = calculate_perplexity(val_sentences, model_type='bigram', k=k)
    k_experiments.append({'Valor de k': k, 'Perplejidad Bigrama': pp_k})

df_k = pd.DataFrame(k_experiments)
print(df_k.to_string(index=False))


Mejor modelo según validación: Bigram
Perplejidad final en el conjunto de Prueba: 846.93

 Valor de k  Perplejidad Bigrama
      1.000          2140.766951
      0.500          1536.451175
      0.100           810.283126
      0.010           491.633718
      0.001           471.393404
